# 🔧 YOLOv8 — Herramientas + EPP HydroAlia
Entrena un modelo que detecta herramientas Y EPP en el mismo frame
para validar que el EPP sea el correcto para cada tarea.

**Runtime: T4 GPU recomendado**

In [ ]:
!pip install roboflow ultralytics -q
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Descargar datasets y combinarlos
from roboflow import Roboflow
import os, shutil, yaml

rf = Roboflow(api_key="duM6tN36cqIVXMbBkawu")

# Dataset 1: Construction Safety (EPP - ya lo tenemos)
ppe = rf.workspace("guillermos-workspace-us1kv").project("construction-safety-gdvov-ckf5v").version(1)
ppe_data = ppe.download("yolov8", location="/content/datasets/ppe")

# Dataset 2: Power Tools Detection (herramientas eléctricas)
tools = rf.workspace("roboflow-universe-projects").project("power-tools-detection").version(1)
tools_data = tools.download("yolov8", location="/content/datasets/tools")

print("Datasets descargados ✓")

In [ ]:
# Combinar ambos datasets
import os, shutil, yaml

COMBINED = "/content/datasets/combined"
for split in ["train", "valid", "test"]:
    for sub in ["images", "labels"]:
        os.makedirs(f"{COMBINED}/{split}/{sub}", exist_ok=True)

# Copiar imágenes y labels de ambos datasets
for dataset_path in ["/content/datasets/ppe", "/content/datasets/tools"]:
    for split in ["train", "valid", "test"]:
        for sub in ["images", "labels"]:
            src = f"{dataset_path}/{split}/{sub}"
            dst = f"{COMBINED}/{split}/{sub}"
            if os.path.exists(src):
                for f in os.listdir(src):
                    shutil.copy(f"{src}/{f}", f"{dst}/{f}")

# Leer clases de ambos datasets
all_classes = []
for dataset_path in ["/content/datasets/ppe", "/content/datasets/tools"]:
    with open(f"{dataset_path}/data.yaml") as f:
        data = yaml.safe_load(f)
        for cls in data.get("names", []):
            if cls not in all_classes:
                all_classes.append(cls)

# Crear data.yaml combinado
combined_yaml = {
    "train": f"{COMBINED}/train/images",
    "val":   f"{COMBINED}/valid/images",
    "test":  f"{COMBINED}/test/images",
    "nc":    len(all_classes),
    "names": all_classes
}
with open(f"{COMBINED}/data.yaml", "w") as f:
    yaml.dump(combined_yaml, f)

print(f"Dataset combinado: {len(all_classes)} clases")
print("Clases:", all_classes)

In [ ]:
# ENTRENAR modelo combinado EPP + Herramientas
from ultralytics import YOLO

model = YOLO("yolov8n.pt")
model.train(
    data=f"{COMBINED}/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    project="/content/drive/MyDrive/ehs-model",
    name="hydroalia-v2",
    patience=15
)
print("Entrenamiento completo ✓")

In [ ]:
# Descargar modelo
from google.colab import files
import glob

pt_files = glob.glob("/content/drive/MyDrive/ehs-model/hydroalia-v2*/weights/best.pt")
print("Modelo guardado en:", pt_files)
if pt_files:
    files.download(pt_files[0])

## Después de descargar best.pt:
1. Copiar a `backend/models/hydroalia-v2.pt`
2. En `.env` cambiar: `MODEL_PATH=models/hydroalia-v2.pt`
3. Reiniciar uvicorn

El sistema ahora detecta herramientas + EPP y valida combinaciones.